### 🚀 Treinamento de modelo Faster R-CNN com backbone CustomDeepCNN e labels no formato YOLO

#### ⚙️ Configuração do dispositivo
- Seleciona automaticamente GPU se disponível, senão utiliza CPU.

#### 📂 Carregamento da configuração YAML
- Carrega os diretórios `train`, `val`, `test`, número de classes (`nc`) e os nomes das classes.

#### 📁 Dataset personalizado (YOLO)
- Classe `YoloDataset`:
  - Lê imagens e rótulos `.txt` no formato YOLO.
  - Filtra imagens sem anotações válidas.
  - Converte para `boxes` absolutas e tensores.
- Aplica transformações (`Resize` + `ToTensor`).
- Cria `DataLoaders` para treino, validação e teste.

#### 🧠 Modelo com backbone personalizado
- Define `CustomDeepCNNBackbone`: uma CNN profunda com convoluções, `ReLU`, e `MaxPool`.
- Usa `AnchorGenerator` e `MultiScaleRoIAlign` para gerar o `Faster R-CNN` com este backbone.
- Número de classes: `nc + 1` (inclui classe de fundo).
- Envia o modelo para o dispositivo.

#### 🔧 Otimização
- Otimizador: `Adam` com `lr = 1e-4`.

#### 🔁 Treinamento com early stopping
- Treina até **150 épocas**, com `early stopping` baseado em `loss` médio.
- A cada batch, imprime:
  - `Loss total`
  - Componentes da loss: classificação, regressão de bounding boxes, objectness e RPN box.
- Se o `loss médio` melhorar significativamente (mais de 0.1), salva o modelo.
- Para se não houver melhoria durante `10 épocas` seguidas.

#### 💾 Salvamento do modelo
- Modelo salvo como `CustomDeepCNNBackbone_yolo_label.pth` quando há melhoria no desempenho.


In [1]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import os
from PIL import Image
import yaml
import matplotlib.pyplot as plt
import cv2
from collections import OrderedDict
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

print("🔧 Inicializando dispositivo...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo utilizado: {device}")

print("📂 Carregando configuração YAML...")
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

train_img_dir = dataset_config["train"]
val_img_dir = dataset_config["val"]
test_img_dir = dataset_config["test"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]
print(f"✅ Dataset carregado: {len(class_names)} classes")

def load_yolo_labels(label_path, img_width, img_height):
    boxes, labels = [], []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, xc, yc, w, h = map(float, line.strip().split())
            x_min = (xc - w / 2) * img_width
            y_min = (yc - h / 2) * img_height
            x_max = (xc + w / 2) * img_width
            y_max = (yc + h / 2) * img_height
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(int(cls))
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

def replace_extension(file):
    return os.path.splitext(file)[0] + ".txt"

class YoloDataset(Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = img_dir.replace("images", "labels")
        self.transforms = transforms
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(('jpg', 'png', 'jpeg'))]

        # Filtrar imagens sem labels
        filtered_imgs = []
        for img in self.imgs:
            label_path = os.path.join(self.label_dir, replace_extension(img))
            if os.path.exists(label_path):
                with open(label_path, 'r') as file:
                    if file.read().strip():
                        filtered_imgs.append(img)
        self.imgs = filtered_imgs

        print(f"📚 Dataset inicializado: {img_dir} com {len(self.imgs)} imagens válidas (com labels)")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        label_path = os.path.join(self.label_dir, replace_extension(self.imgs[idx]))

        img = Image.open(img_path).convert("RGB")
        img_width, img_height = img.size

        boxes, labels = load_yolo_labels(label_path, img_width, img_height)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}

        if self.transforms:
            img = self.transforms(img)

        return img, target

transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor()
])


train_dataset = YoloDataset(train_img_dir, transform)
val_dataset = YoloDataset(val_img_dir, transform)
test_dataset = YoloDataset(test_img_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

class CustomDeepCNNBackbone(nn.Module):
    def __init__(self):
        super(CustomDeepCNNBackbone, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=2), nn.ReLU(),  # [B, 16, H/2, W/2]
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # [B, 32, H/4, W/4]

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # [B, 128, H/8, W/8]

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1), nn.ReLU(),
        )
        self.out_channels = 512

    def forward(self, x):
        return OrderedDict([("0", self.features(x))])


print("🔨 Construindo modelo...")
backbone = CustomDeepCNNBackbone()
anchor_generator = AnchorGenerator(sizes=((32, 64, 128, 256, 512),), aspect_ratios=((0.5, 1.0, 2.0),))
roi_pooler = MultiScaleRoIAlign(featmap_names=['0'], output_size=7, sampling_ratio=2)
model = torchvision.models.detection.FasterRCNN(backbone, num_classes=nc + 1, rpn_anchor_generator=anchor_generator, box_roi_pool=roi_pooler)
model.to(device)
print("✅ Modelo pronto para treino")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("🚀 Iniciando treinamento...")
model.train()

prev_loss = float('inf')
patience = 10
no_improve_epochs = 0
max_epochs = 150  # aumenta para ter mais chance de aprender

for epoch in range(max_epochs):
    model.train()
    total_loss = 0.0
    total_cls_loss = 0.0
    total_box_loss = 0.0
    total_obj_loss = 0.0
    total_rpn_box_loss = 0.0

    for batch_idx, (imgs, targets) in enumerate(train_loader, 1):
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        total_cls_loss += loss_dict['loss_classifier'].item()
        total_box_loss += loss_dict['loss_box_reg'].item()
        total_obj_loss += loss_dict['loss_objectness'].item()
        total_rpn_box_loss += loss_dict['loss_rpn_box_reg'].item()

        print(f"📈 Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, "
              f"Loss: {losses.item():.4f} "
              f"(cls: {loss_dict['loss_classifier']:.4f}, "
              f"box: {loss_dict['loss_box_reg']:.4f}, "
              f"obj: {loss_dict['loss_objectness']:.4f}, "
              f"rpn: {loss_dict['loss_rpn_box_reg']:.4f})")

    avg_loss = total_loss / len(train_loader)
    print(f"\n🎯 Epoch {epoch+1} completa. "
          f"Loss Total: {avg_loss:.4f} | "
          f"cls: {total_cls_loss:.4f}, box: {total_box_loss:.4f}, "
          f"obj: {total_obj_loss:.4f}, rpn: {total_rpn_box_loss:.4f}\n")

    if avg_loss + 0.1 < prev_loss:
        print("✅ Melhorando, continuando...")
        prev_loss = avg_loss
        no_improve_epochs = 0
        torch.save(model.state_dict(), "models/CustomDeepCNNBackbone.pth")
        print("💾 Modelo salvo!")
    else:
        no_improve_epochs += 1
        print("⚠️ Não houve melhoria.")
        if no_improve_epochs >= patience:
            print("🛑 Parando o treinamento por falta de melhoria.")
            break


🔧 Inicializando dispositivo...
✅ Dispositivo utilizado: cuda
📂 Carregando configuração YAML...
✅ Dataset carregado: 9 classes
📚 Dataset inicializado: C:\\Users\\Jorge Cunha\\Desktop\\Python\\AI_Detection_Cars\\AAU2\\DetectObjects_JC\\dataset\\train\\images com 1258 imagens válidas (com labels)
📚 Dataset inicializado: C:\\Users\\Jorge Cunha\\Desktop\\Python\\AI_Detection_Cars\\AAU2\\DetectObjects_JC\\dataset\\val\\images com 158 imagens válidas (com labels)
📚 Dataset inicializado: C:\\Users\\Jorge Cunha\\Desktop\\Python\\AI_Detection_Cars\\AAU2\\DetectObjects_JC\\dataset\\test\\images com 167 imagens válidas (com labels)
🔨 Construindo modelo...
✅ Modelo pronto para treino
🚀 Iniciando treinamento...
📈 Epoch 1, Batch 1/315, Loss: 13.0913 (cls: 2.3080, box: 0.0004, obj: 0.6931, rpn: 10.0897)
📈 Epoch 1, Batch 2/315, Loss: 6.2665 (cls: 2.2634, box: 0.0000, obj: 0.6929, rpn: 3.3102)
📈 Epoch 1, Batch 3/315, Loss: 8.2486 (cls: 2.2135, box: 0.0000, obj: 0.6928, rpn: 5.3422)
📈 Epoch 1, Batch 4/31

### 📊 Avaliação de modelo Faster R-CNN com backbone `CustomDeepCNN` e anotações no formato YOLO

#### ⚙️ Dispositivo
- Seleciona GPU (`cuda`) se disponível, caso contrário utiliza CPU.

#### 📂 Configuração do Dataset
- Carrega do `dataset.yaml`:
  - Diretório de validação.
  - Número total de classes (`nc`).
  - Lista com os nomes das classes.

#### 🗂️ Dataset personalizado (`YoloDataset`)
- Lê imagens e respetivos ficheiros `.txt` no formato YOLO.
- Converte bounding boxes normalizadas para coordenadas absolutas.
- Aplica redimensionamento (`128x128`) e normalização.
- Apenas imagens com ficheiros de rótulo válidos são mantidas.

#### 🧠 Definição do Modelo
- Define `CustomDeepCNNBackbone`, uma rede convolucional com várias camadas `Conv2D`, `ReLU`, e `MaxPool`.
- Backbone retorna `OrderedDict` com um único mapa de características ("0").
- Instancia modelo `Faster R-CNN` com:
  - Anchors multiescala e multiproporção.
  - Pooling ROI (`MultiScaleRoIAlign`).
  - `nc + 1` classes (inclui classe de fundo).
- Carrega pesos previamente treinados.

#### 📈 Avaliação
- Para cada imagem:
  - Extrai predições do modelo com confiança acima do limiar (`threshold`, default = 0.0).
  - Compara com as classes reais.
  - Marca presença/ausência de cada classe.
- Calcula e imprime:
  - **Precisão**
  - **Revocação**
  - **F1-score**

#### 🚀 Execução
- A função `evaluate_model` é chamada com o `val_loader`.
- Resultados são impressos no terminal.


In [4]:
import torch
from sklearn.metrics import precision_score, recall_score, f1_score
from torchvision import transforms
from torch.utils.data import DataLoader
import yaml
from PIL import Image
import os
import torch.nn as nn
from collections import OrderedDict
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

# 🔧 Dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Usando dispositivo: {device}")

# 📂 Carregar configuração do dataset
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

val_img_dir = dataset_config["val"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]

# Função para carregar labels YOLO
def load_yolo_labels(label_path, img_width, img_height):
    boxes, labels = [], []
    with open(label_path, "r") as f:
        for line in f.readlines():
            cls, xc, yc, w, h = map(float, line.strip().split())
            x_min = (xc - w / 2) * img_width
            y_min = (yc - h / 2) * img_height
            x_max = (xc + w / 2) * img_width
            y_max = (yc + h / 2) * img_height
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(int(cls))
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

# Dataset de validação com resize 128x128
class YoloDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = img_dir.replace("images", "labels")
        self.transforms = transforms
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(('jpg', 'png', 'jpeg'))]

        self.imgs = [
            img for img in self.imgs
            if os.path.exists(os.path.join(self.label_dir, os.path.splitext(img)[0] + ".txt"))
        ]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        label_path = os.path.join(self.label_dir, os.path.splitext(self.imgs[idx])[0] + ".txt")

        img = Image.open(img_path).convert("RGB")
        img_width, img_height = img.size

        boxes, labels = load_yolo_labels(label_path, img_width, img_height)
        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)

        return img, target

# Transforma imagem igual ao treino
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

val_dataset = YoloDataset(val_img_dir, transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# 🔧 Modelo com backbone CustomDeepCNNBackbone
class CustomDeepCNNBackbone(nn.Module):
    def __init__(self):
        super(CustomDeepCNNBackbone, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=2), nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1), nn.ReLU()
        )
        self.out_channels = 512

    def forward(self, x):
        return OrderedDict([("0", self.features(x))])

# Modelo completo
backbone = CustomDeepCNNBackbone()
anchor_gen = AnchorGenerator(sizes=((32, 64, 128, 256, 512),), aspect_ratios=((0.5, 1.0, 2.0),))
roi_pool = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)

model = FasterRCNN(backbone, num_classes=nc + 1, rpn_anchor_generator=anchor_gen, box_roi_pool=roi_pool)
model.load_state_dict(torch.load("models/CustomDeepCNNBackbone.pth", map_location=device))
model.to(device)
model.eval()

# 📊 Função de Avaliação
def evaluate_model(model, dataloader, threshold=0.0):
    y_true_all, y_pred_all = [], []

    with torch.no_grad():
        for imgs, targets in dataloader:
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for target, pred in zip(targets, preds):
                gt_labels = target["labels"].cpu().numpy()
                scores = pred["scores"].cpu().numpy()
                pred_labels = pred["labels"].cpu().numpy()

                # Filtrar por limiar de confiança
                pred_labels = pred_labels[scores > threshold] if len(scores) > 0 else []

                for cls in range(nc):
                    y_true_all.append(1 if cls in gt_labels else 0)
                    y_pred_all.append(1 if cls in pred_labels else 0)

    # Métricas agregadas
    precision = precision_score(y_true_all, y_pred_all, zero_division=0)
    recall = recall_score(y_true_all, y_pred_all, zero_division=0)
    f1 = f1_score(y_true_all, y_pred_all, zero_division=0)

    print(f"\n📌 Avaliação com limiar > {threshold}")
    print(f"🎯 Precisão: {precision:.3f}")
    print(f"🎯 Revocação: {recall:.3f}")
    print(f"🎯 F1-score: {f1:.3f}")

# 🚀 Executar Avaliação
evaluate_model(model, val_loader, threshold=0.0)


🖥️ Usando dispositivo: cuda


C:\Users\Jorge Cunha\AppData\Local\Temp\ipykernel_19796\1927560234.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("models/faster_rcnn


📌 Avaliação com limiar > 0.0
🎯 Precisão: 0.329
🎯 Revocação: 0.108
🎯 F1-score: 0.162
